# Paper reproduction — *The Benchmark Chooses the Winner*

**Canonical single-notebook entrypoint.** Run All to regenerate every paper result family. This notebook
orchestrates the committed paper scripts (so its logic cannot drift from the paper) and renders all tables
and figures from the JSON those scripts emit.

Two modes (set the `REPRO_MODE` env var before launching the kernel):
- `verify_cached` (default): load existing `outputs/nb-smollm3-guard/summary_*.json` and render fast.
- `recompute`: run each paper script to (re)produce the JSON first, then render. **Heavy** — needs a
  GPU for the local guards, an `OPENAI_API_KEY` for the GPT baseline, network for the open guards, and
  time (hours on a laptop; some open-guard/novel runs are GPU-pending per the paper).

> Honesty note: this replaces the older `smollm3_guard_reproduction.ipynb` companion *demo* as the paper
> path. Full byte-exact `recompute` also depends on upstream Hugging Face datasets that are not revision
> pinned (see the repo README caveats).


## 0 · Preflight & mode


In [ ]:
import os, json, subprocess, sys
import pandas as pd
try:
    import torch; _cuda = torch.cuda.is_available(); _mps = getattr(torch.backends,'mps',None) and torch.backends.mps.is_available()
except Exception:
    _cuda=_mps=False
NB_DIR = os.getcwd()
RESEARCH = NB_DIR if os.path.isdir(os.path.join(NB_DIR,'scripts')) else os.path.dirname(NB_DIR)
OUT = os.path.join(RESEARCH,'notebooks','outputs','nb-smollm3-guard')
MODE = os.environ.get('REPRO_MODE','verify_cached').lower()
assert MODE in ('verify_cached','recompute'), MODE
HAVE_OPENAI = bool(os.environ.get('OPENAI_API_KEY'))
print(f'MODE={MODE} | RESEARCH={RESEARCH}')
print(f'device: cuda={_cuda} mps={_mps} | OPENAI_API_KEY={"set" if HAVE_OPENAI else "MISSING"}')
if MODE=='recompute' and not _cuda:
    print('WARNING: recompute without CUDA is very slow on this box; local-guard scoring may take hours.')
if MODE=='recompute' and not HAVE_OPENAI:
    print('WARNING: no OPENAI_API_KEY — the GPT baseline / re-ground cells will be skipped.')

def run_script(name, env=None):
    if MODE!='recompute':
        print(f'[verify] skipping scripts/{name} (using cached JSON)'); return
    print(f'[recompute] python scripts/{name}  (cwd={RESEARCH})', flush=True)
    p = subprocess.run([sys.executable,'-u',os.path.join('scripts',name)], cwd=RESEARCH,
                       env={**os.environ, **(env or {})})
    if p.returncode!=0: raise RuntimeError(f'scripts/{name} failed (exit {p.returncode})')

def load_json(name, produced_by):
    fp=os.path.join(OUT,name)
    if not os.path.exists(fp):
        raise FileNotFoundError(f'{name} not found. Re-run with REPRO_MODE=recompute, or run scripts/{produced_by} first.')
    return json.load(open(fp))
HEADLINE={}  # collected for the final validation checklist


## 1 · In-house corrected evaluation  (`eval_corrected.py`)
Matched-FPR@0.10, threshold-free AUPRC/AUROC, clean in-dist-only calibration.


In [ ]:
run_script('eval_corrected.py')
s = load_json('summary_corrected.json','eval_corrected.py')
print('calibration:', s.get('calibration',{}).get('clean'))
print('guard batch=1 latency ms:', s.get('guard_batch1_latency_ms'))
mf = s.get('matched_fpr_0.10',{})
display(pd.DataFrame(mf).T.rename_axis('system')) if mf else print(s)


## 2 · Novel / OOD evaluation  (`eval_novel_gaps.py` → `verify_novel.py`)
Aggregate AUPRC over the 3 balanced novel sets: base > tuned guard > Llama-Guard.


In [ ]:
run_script('eval_novel_gaps.py'); run_script('verify_novel.py')
s = load_json('summary_novel_full.json','verify_novel.py')
rows=[{'system':k,'AUPRC':v.get('auprc'),'CI':v.get('ci'),'best_F1':v.get('best_f1')} for k,v in s.items()]
HEADLINE['novel_base_auprc']=s.get('base',{}).get('auprc'); HEADLINE['novel_guard_auprc']=s.get('guard',{}).get('auprc'); HEADLINE['novel_llama_auprc']=s.get('llama-guard',{}).get('auprc')
display(pd.DataFrame(rows).set_index('system'))


## 3 · Base-vs-tuned decomposition  (`score_base_inhouse.py` → `recompute_base_vs_tuned.py`)


In [ ]:
run_script('score_base_inhouse.py'); run_script('recompute_base_vs_tuned.py')
s = load_json('base_vs_tuned_clean.json','recompute_base_vs_tuned.py')
HEADLINE['base_vs_tuned_delta']=s.get('pooled',{}).get('delta')
print('pooled:', s.get('pooled')); print('base in-house AUPRC:', s.get('base_inhouse_auprc',{}).get('pooled'))
pb=s.get('per_benchmark',{})
display(pd.DataFrame(pb).T[['base','tuned','delta']]) if pb else None


## 4 · Mortgage hardened case study  (`eval_mortgage_hard.py`)
Loads the committed 334-row `guard_benchmark_hard.jsonl` directly (the superseded synthetic builders are NOT used).


In [ ]:
run_script('eval_mortgage_hard.py')
s = load_json('summary_mortgage_hard.json','eval_mortgage_hard.py')
sy=s.get('systems',{})
cols=['auprc','opt_f1','acc','recall','precision','fpr']
df=pd.DataFrame({k:{c:v.get(c) for c in cols} for k,v in sy.items()}).T
HEADLINE['mortgage_sft_auprc']=sy.get('mortgage-sft',{}).get('auprc')
print(f"n={s.get('n')} flag={s.get('flag')} allow={s.get('allow')}"); display(df)


## 5 · GPT baseline re-ground (abstain policy)  (`reground_gpt_inhouse.py`)
Recomputes the in-house guard-vs-GPT F1 with the paper's *abstain* error policy (not fail-closed).


In [ ]:
if MODE=='recompute' and not HAVE_OPENAI:
    print('skipped: no OPENAI_API_KEY')
else:
    run_script('reground_gpt_inhouse.py')
    s = load_json('summary_gpt_reground.json','reground_gpt_inhouse.py')
    HEADLINE['gpt_delta_f1']=s.get('delta_f1_guard_minus_gpt')
    print(f"abstained={s.get('n_abstain')}/{s.get('n_rows')}  API errors={s.get('n_api_errors')}  preds changed vs old fail-closed={s.get('gpt_new_vs_old_changed_preds')}")
    print(f"gpt FPR old(fail-closed)={s['gpt_old_failclosed']['fpr']:.3f} -> new(abstain)={s['gpt_abstain']['fpr']:.3f}")
    print(f"Delta F1 (guard-gpt)={s['delta_f1_guard_minus_gpt']:+.3f} CI={s['delta_f1_ci']} McNemar p={s['mcnemar_p']:.3f}")


## 6 · Figures (drawn from the JSON above, not inline constants)


In [ ]:
import matplotlib.pyplot as plt
# novel aggregate AUPRC
try:
    nv=load_json('summary_novel_full.json','verify_novel.py')
    order=['base','guard','llama-guard']; vals=[nv[k]['auprc'] for k in order]
    fig,ax=plt.subplots(figsize=(4,2.6)); ax.bar(order,vals,color=['#0072B2','#E69F00','#009E73'])
    [ax.text(i,v+.01,f'{v:.3f}',ha='center',fontsize=8) for i,v in enumerate(vals)]
    ax.set_ylim(0,1); ax.set_title('Novel aggregate AUPRC (from JSON)'); plt.tight_layout(); plt.show()
except Exception as e: print('novel fig skipped:',e)
# mortgage hardened AUPRC
try:
    mh=load_json('summary_mortgage_hard.json','eval_mortgage_hard.py')['systems']
    order=[k for k in ('base','mortgage-sft','general-guard') if k in mh]; vals=[mh[k]['auprc'] for k in order]
    fig,ax=plt.subplots(figsize=(4,2.6)); ax.bar(order,vals,color='#0072B2')
    [ax.text(i,v+.01,f'{v:.3f}',ha='center',fontsize=8) for i,v in enumerate(vals)]
    ax.set_ylim(0,1); ax.set_title('Hardened mortgage AUPRC (from JSON)'); plt.tight_layout(); plt.show()
except Exception as e: print('mortgage fig skipped:',e)


## 7 · Validation checklist
Compares the regenerated numbers against the paper's reported headline values.


In [ ]:
EXPECTED={'novel_base_auprc':0.886,'novel_guard_auprc':0.781,'novel_llama_auprc':0.701,
          'base_vs_tuned_delta':0.081,'mortgage_sft_auprc':0.924,'gpt_delta_f1':0.010}
rows=[]
for k,exp in EXPECTED.items():
    got=HEADLINE.get(k)
    ok = (got is not None) and abs(float(got)-exp)<=0.03
    rows.append({'metric':k,'expected':exp,'got':(round(float(got),3) if got is not None else None),'PASS(±0.03)':ok})
chk=pd.DataFrame(rows).set_index('metric'); display(chk)
n_ok=int(chk['PASS(±0.03)'].sum()); print(f'{n_ok}/{len(chk)} headline numbers match (tol 0.03).')
if MODE=='verify_cached': print('(verify_cached: numbers reflect the committed/produced JSON, not a fresh recompute.)')
